In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mshrestha/plant-disease-augmented-dataset")

print("Path to dataset files:", path)

100%|██████████| 8.78G/8.78G [01:29<00:00, 105MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mshrestha/plant-disease-augmented-dataset/versions/1


In [3]:
cd /root/.cache/kagglehub/datasets/mshrestha/plant-disease-augmented-dataset/versions/1

/root/.cache/kagglehub/datasets/mshrestha/plant-disease-augmented-dataset/versions/1


In [4]:
ls

APPLE_HEALTHY/               RICE_LEAF_BLAST/
APPLE_ROT/                   RICE_LEAF_BLIGHT/
APPLE_RUST/                  RICE_LEAF_BROWN_SPOT/
APPLE_SCAB/                  STRAWBERRY_HEALTHY/
BANANA_HEALTHY/              STRAWBERRY_LEAF_SCORCH/
BANANA_PANAMA/               TEA_ALGAL_SPOT/
BANANA_SIGATOKA/             TEA_BROWN_BLIGHT/
CORN_HEALTHY/                TEA_HEALTHY/
CORN_LEAF_BLIGHT/            TEA_RED_LEAF_SPOT/
CORN_LEAF_GRAY_SPOT/         TOMATO_BACTERIAL_SPOT/
CORN_LEAF_RUST/              TOMATO_EARLY_BLIGHT/
PEPPER_BELL_BACTERIAL_SPOT/  TOMATO_HEALTHY/
PEPPER_BELL_HEALTHY/         TOMATO_LATE_BLIGHT/
POTATO_EARLY_BLIGHT/         TOMATO_LEAF_MOLD/
POTATO_HEALTHY/              TOMATO_MOSAIC_VIRUS/
POTATO_LATE_BLIGHT/          TOMATO_SEPTORIA_LEAF_SPOT/
RICE_HEALTHY/                TOMATO_TARGET_SPOT/


In [7]:
import os
import shutil

SOURCE_DIR = "/root/.cache/kagglehub/datasets/mshrestha/plant-disease-augmented-dataset/versions/1"
TARGET_DIR = "/content/crop_dataset"

classes = [
    "RICE_HEALTHY",
    "RICE_LEAF_BLAST",
    "RICE_LEAF_BLIGHT",
    "RICE_LEAF_BROWN_SPOT",
    "CORN_HEALTHY",
    "CORN_LEAF_RUST",
    "PEPPER_BELL_HEALTHY",
    "PEPPER_BELL_BACTERIAL_SPOT"
]

os.makedirs(TARGET_DIR, exist_ok=True)

for c in classes:
    src = os.path.join(SOURCE_DIR, c)
    dst = os.path.join(TARGET_DIR, c)

    if os.path.exists(src):
        shutil.copytree(src, dst)
    else:
        print(f"Missing folder: {c}")

print("Filtered dataset created.")

Filtered dataset created.


In [8]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_ds = image_dataset_from_directory(
    "/content/crop_dataset",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

val_ds = image_dataset_from_directory(
    "/content/crop_dataset",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

class_names = train_ds.class_names
print(class_names)

Found 33089 files belonging to 8 classes.
Using 26472 files for training.
Found 33089 files belonging to 8 classes.
Using 6617 files for validation.
['CORN_HEALTHY', 'CORN_LEAF_RUST', 'PEPPER_BELL_BACTERIAL_SPOT', 'PEPPER_BELL_HEALTHY', 'RICE_HEALTHY', 'RICE_LEAF_BLAST', 'RICE_LEAF_BLIGHT', 'RICE_LEAF_BROWN_SPOT']


In [9]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
NUM_CLASSES = len(class_names)

base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
828/828 ━━━━━━━━━━━━━━━━━━━━ 1398s 2s/step - accuracy: 0.8066 - loss: 0.5313 - val_accuracy: 0.9326 - val_loss: 0.1846
Epoch 2/5
828/828 ━━━━━━━━━━━━━━━━━━━━ 1454s 2s/step - accuracy: 0.9310 - loss: 0.1880 - val_accuracy: 0.9408 - val_loss: 0.1634
Epoch 3/5
828/828 ━━━━━━━━━━━━━━━━━━━━ 1421s 2s/step - accuracy: 0.9419 - loss: 0.1627 - val_accuracy: 0.9471 - val_loss: 0.1434
Epoch 4/5
828/828 ━━━━━━━━━━━━━━━━━━━━ 1401s 2s/step - accuracy: 0.9459 - loss: 0.1495 - val_accuracy: 0.9488 - val_loss: 0.1397
Epoch 5/5
828/828 ━━━━━━━━━━━━━━━━━━━━ 1418s 2s/step - accuracy: 0.9458 - loss: 0.1434 - val_accuracy: 0.9528 - val_loss: 0.1304


In [17]:
import os
os.makedirs("/content/export", exist_ok=True)

model.save("/content/export/model.keras")
print("Saved:", "/content/export/model.keras")

Saved: /content/export/model.keras


In [18]:
import json, os
os.makedirs("/content/export", exist_ok=True)

labels = {i: name for i, name in enumerate(class_names)}
with open("/content/export/labels.json", "w") as f:
    json.dump(labels, f, indent=2)

In [19]:
from google.colab import files
files.download("/content/export/model.keras")
files.download("/content/export/labels.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>